In [3]:
"""Importing libraries"""
import time
import warnings
import os
import pywt # pylint: disable=import-error
import numpy as np
from osgeo import gdal
from skimage.exposure import match_histograms # pylint: disable=import-error
from skimage import color # pylint: disable=import-error
import spectral as spy # pylint: disable=import-error
from scipy.ndimage import uniform_filter, variance
from sklearn.decomposition import PCA # pylint: disable=import-error
from sklearn.preprocessing import StandardScaler # pylint: disable=import-error
warnings.filterwarnings("ignore")
def stretch(bands, lower_percent=1, higher_percent=98):
    """Contrast stretch function, used to improve contrast and brightness of the images"""
    np.ma.array(bands, mask=np.isnan(bands))
    out = np.zeros_like(bands)
    a_1 = 0
    b_1 = 255
    c_1 = np.percentile(bands, lower_percent)
    d_1 = np.percentile(bands, higher_percent)
    t_1 = a_1 + (bands - c_1) * (b_1 - a_1)/(d_1 - c_1)
    t_1[t_1<a_1] = a_1
    t_1[t_1>b_1] = b_1
    out = t_1
    return out.astype(np.uint8)
COUNT = 1
def increment():
    global COUNT
    COUNT = COUNT + 1
def lee_filter(img, size):
    """Lee filter function for speckle filtering"""
    img_mean = uniform_filter(img, (size, size))
    img_sqr_mean = uniform_filter(img**2, (size, size))
    img_variance = img_sqr_mean - img_mean**2
    overall_variance = variance(img)
    img_weights = img_variance / (img_variance + overall_variance)
    img_output = img_mean + img_weights * (img - img_mean)
    increment()
    return img_output
def pct_image(img, eigs):
    """Principal components calculated from spectral package"""
    pc_1 = spy.principal_components(img)
    ch_1 = pc_1.reduce(eigs = eigs)
    img_pc = ch_1.transform(img)
    return img_pc
def span_image(img):
    """Build the span image"""
    img_sp = img[:,:,0]**2 + 2*abs(img[:,:,1]) + img[:,:,3]**2
    return img_sp
def image_fusion_pca(image1, image2, nodata_val=0):
    """"Function of principal component analysis for image fusion"""
    rows, columns, bands = image1.shape
    out_image = np.zeros_like(image1)
    valid_mask = ~(
        np.all(image1 == nodata_val, axis=2) | (image2 == nodata_val)
    )
    valid_pixels = np.where(valid_mask.flatten())[0]
    if len(valid_pixels) == 0:
        return out_image
    reshaped_opt = image1.reshape(-1, bands)
    sar_flat = image2.flatten()
    opt_valid = reshaped_opt[valid_pixels]
    sar_valid = sar_flat[valid_pixels]
    scaler = StandardScaler()
    opt_scaled = scaler.fit_transform(opt_valid)
    sar_std = (sar_valid - np.mean(sar_valid)) / (np.std(sar_valid) + 1e-8)
    pca = PCA(n_components=bands)
    opt_pca = pca.fit_transform(opt_scaled)
    opt_pca[:, 0] = sar_std
    opt_recon = pca.inverse_transform(opt_pca)
    opt_fused = scaler.inverse_transform(opt_recon)
    fused_reshaped = reshaped_opt.copy()
    fused_reshaped[valid_pixels] = opt_fused
    out_image = fused_reshaped.reshape(rows, columns, bands)
    out_image = np.clip(out_image, 0, 255).astype(np.uint8)
    return out_image
def channelTransform(ch1, ch2, shape):
    """Function represents channel transformation for DWT"""
    cooef1 = pywt.dwt2(ch1, 'db5', mode = 'periodization')
    cooef2 = pywt.dwt2(ch2, 'db5', mode = 'periodization')
    ca1, (ch1, cv1, cd1) = cooef1
    ca2, (ch2, cv2, cd2) = cooef2
    ca_t = (ca1 * 0.7 + ca2 * 0.3)
    ch_t = (ch1 * 0.7 + ch2 * 0.3)
    cv_t = (cv1 * 0.7 + cv2 * 0.3)
    cd_t = (cd1 * 0.7 + cd2 * 0.3)
    fin_coc = ca_t, (ch_t, cv_t, cd_t)
    out_imagec = pywt.idwt2(fin_coc, 'db5', mode = 'periodization')
    return out_imagec
def dwt_fusion(i_1, i_2):
    """Function of discrete wavelet transform for image fusion"""
    ir_1 = i_1[:,:,0]
    ir_2 = i_2.copy()
    ig_1 = i_1[:,:,1]
    ig_2 = i_2.copy()
    ib_1 = i_1[:,:,2]
    ib_2 = i_2.copy()
    shape = (i_1.shape[1], i_1.shape[0])
    out_imager = channelTransform(ir_1, ir_2, shape)
    out_imageg = channelTransform(ig_1, ig_2, shape)
    out_imageb = channelTransform(ib_1, ib_2, shape)
    out_image = i_1.copy()
    out_image[:,:,0] = out_image[:,:,1] = out_image[:,:,2] = 0
    out_image[:,:,0] = out_imager
    out_image[:,:,1] = out_imageg
    out_image[:,:,2] = out_imageb
    out_image = np.multiply(np.divide(out_image - np.min(out_image),\
                                     (np.max(out_image) - np.min(out_image))),255)
    out_image = out_image.astype(np.uint8)
    return out_image
def create_multiband_geotiff(array, out_name, proj, geo, nodata=0,\
                             out_format = gdal.GDT_Byte, verbose = False):
    """Generating multiband geotiff image, along with following
    details like projection, geotransform, and nodata values"""
    driver = gdal.GetDriverByName('GTiff')
    if len(array.shape) == 2:
        array = array[np.newaxisw, ...]
    os.makedirs(os.path.dirname(os.path.abspath(out_name)), exist_ok = True)
    dataset = driver.Create(out_name, array.shape[2], array.shape[1],\
                            array.shape[0], out_format)
    if proj is not None:
        dataset.SetProjection(proj)
    if geo is not None:
        dataset.SetGeoTransform(geo)
    if nodata is None:
        for i, image in enumerate(array, 1):
            dataset.GetRasterBand(i).WriteArray(image)
        del dataset
    else:
        for i, image in enumerate(array, 1):
            dataset.GetRasterBand(i).WriteArray(image)
            dataset.GetRasterBand(i).SetNoDataValue(nodata)
        del dataset
def fusion(rgb, sar, method):
    """Types of fusion methods"""
    red_channel = rgb[:,:,0]
    green_channel = rgb[:,:,1]
    blue_channel = rgb[:,:,2]
    image = None
    if method == 'DWT':
        image = dwt_fusion(rgb,sar)
    if method == 'PCA':
        image = image_fusion_pca(rgb,sar)
    return image
def color_sar(image_path1, image_path2, method):
    """colorizes SAR data using co-located RGB image"""
    span_or_pca = 'span'
    filtersize = 3
    nodata = 0
    add_4th = False
    sar_img = gdal.Open(image_path1)
    proj = sar_img.GetProjection()
    geo = sar_img.GetGeoTransform()
    sar_img = sar_img.ReadAsArray()
    sar_img = np.swapaxes(sar_img, 0 ,2)
    sar_img = np.swapaxes(sar_img, 0 ,1)
    if sar_img.shape[2] > 1:
        if span_or_pca == 'span':
            span_img = span_image(sar_img)
        elif span_or_pca == 'pca':
            span_img = pct_image(sar_img, 0)
        else:
            print("Choose 'span' or 'PCA' for span_or_pca ")
    else:
        span_img = sar_img[0,:,:]
    rgb_file = image_path2
    dataset = gdal.Open(rgb_file)
    rgb_img = dataset.ReadAsArray().transpose(1, 2, 0)
    rgb_img = rgb_img.astype(np.float32)
    sar_img_hgm = match_histograms(span_img, rgb_img[:,:,2])
    lee_filt_img = lee_filter(sar_img_hgm, filtersize)
    eo_sar_fusion = fusion(rgb_img, lee_filt_img, method = method)
    eo_sar_fusion = np.swapaxes(eo_sar_fusion, 1, 0)
    eo_sar_fusion = np.swapaxes(eo_sar_fusion, 2, 0)
    output_image = "Fused_image.tif"
    if add_4th is True:
        eo_sar_fusion = np.stack([eo_sar_fusion[0,:,:], eo_sar_fusion[1,:,:],\
                                  eo_sar_fusion[2,:,:], eo_sar_fusion[0,:,:]])
    create_multiband_geotiff(eo_sar_fusion, output_image, proj, geo, nodata = nodata)
def run_image_fusion(algo_details, image_path1, image_path2):
    """run_image_fusion is a main function"""
    start_time = time.time()
    print("[0%] starting image fusion")
    image = gdal.Open(image_path1)
    print("[10%] loading input images")
    image = image.ReadAsArray()
    if image.shape[0] == 4:
        pass
    else:
        image_path1, image_path2 = image_path2, image_path1
    print("[30%] reading both SAR and optical images")
    color_sar(image_path1, image_path2, algo_details)
    print("[100%] completed, fused image generated")
    end_time = time.time()
    print("Computational time in seconds:", end_time - start_time)
if __name__ == "__main__":
    image_path1 = 'SAR.tif' #SAR input image
    image_path2 = 'RGB.tif' #Optical input image
    algo_details = 'DWT' #Selection of methods (DWT and PCA)
    run_image_fusion(algo_details, image_path1, image_path2)

[0%] starting image fusion
[10%] loading input images
[30%] reading both SAR and optical images
[100%] completed, fused image generated
Computational time in seconds: 1.4264795780181885
